# Skincare Product Recommender System
## Step 1: Data Preprocessing & Merging

**Objective:** Merge all review data, combine with product metadata, clean and preprocess for recommendation system.

**Inputs:**
- `product_info.csv` - Sephora product metadata
- `reviews_*.csv` (5 files) - User reviews split by range

**Output:**
- `output/merged_data.csv` - Clean merged dataset ready for recommendation
- `output/data_summary.json` - Statistics about processed data

---

## 1. Import Libraries and Load Data

In [1]:
import csv
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load product metadata
print("="*60)
print("LOADING PRODUCT METADATA")
print("="*60)

products_df = pd.read_csv("input/product_info.csv", quoting=csv.QUOTE_MINIMAL, quotechar='"')
print(f"\n✅ Products DataFrame shape: {products_df.shape}")
print(f"Columns: {list(products_df.columns)[:10]}... ({len(products_df.columns)} total)")
print("\nFirst row sample:")
display(products_df[['product_id', 'product_name', 'brand_name', 'price_usd', 'rating', 'reviews']].head(1))

LOADING PRODUCT METADATA

✅ Products DataFrame shape: (8494, 27)
Columns: ['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count', 'rating', 'reviews', 'size', 'variation_type', 'variation_value']... (27 total)

First row sample:


,product_id,product_name,brand_name,price_usd,rating,reviews
0,P473671,Fragrance Discovery Set,19-69,35.0,3.6364,11.0


In [3]:
# Load and merge all review files
print("\n" + "="*60)
print("LOADING REVIEW DATA")
print("="*60)

review_files = [
    'input/reviews_0-250.csv',
    'input/reviews_250-500.csv',
    'input/reviews_500-750.csv',
    'input/reviews_750-1250.csv',
    'input/reviews_1250-end.csv'
]

reviews_list = []
for file_path in review_files:
    try:
        df = pd.read_csv(file_path, quoting=csv.QUOTE_MINIMAL, quotechar='"', index_col=0)
        reviews_list.append(df)
        print(f"✅ {file_path}: {df.shape[0]:,} rows")
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")

# Merge all reviews into one dataframe
reviews_df = pd.concat(reviews_list, ignore_index=True)
print(f"\n✅ Total merged reviews: {len(reviews_df):,} rows")
print(f"Columns: {list(reviews_df.columns)[:10]}... ({len(reviews_df.columns)} total)")
print(f"\nRating distribution:")
print(reviews_df['rating'].value_counts().sort_index())


LOADING REVIEW DATA
✅ input/reviews_0-250.csv: 602,130 rows
✅ input/reviews_250-500.csv: 206,725 rows
✅ input/reviews_500-750.csv: 116,262 rows
✅ input/reviews_750-1250.csv: 119,317 rows
✅ input/reviews_1250-end.csv: 49,977 rows

✅ Total merged reviews: 1,094,411 rows
Columns: ['author_id', 'rating', 'is_recommended', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text', 'review_title']... (18 total)

Rating distribution:
rating
1     61223
2     53032
3     81816
4    199389
5    698951
Name: count, dtype: int64


---

## 2. Clean and Preprocess Review Data

In [4]:
print("\n" + "="*60)
print("DATA CLEANING & PREPROCESSING")
print("="*60)

reviews_clean = reviews_df.copy()

# Step 1: Remove invalid ratings
print("\n1️⃣ Cleaning ratings...")
initial_count = len(reviews_clean)

reviews_clean = reviews_clean[reviews_clean['rating'].notna()]
reviews_clean['rating'] = pd.to_numeric(reviews_clean['rating'], errors='coerce')
reviews_clean = reviews_clean[reviews_clean['rating'].notna()]

print(f"  Removed {initial_count - len(reviews_clean):,} rows with invalid ratings")
print(f"  Remaining: {len(reviews_clean):,} rows")

# Step 2: Remove duplicate reviews (same user + product)
print("\n2️⃣ Removing duplicate reviews...")
initial_count = len(reviews_clean)

reviews_clean = reviews_clean.drop_duplicates(subset=['author_id', 'product_id'], keep='first')

print(f"  Removed {initial_count - len(reviews_clean):,} duplicate reviews")
print(f"  Remaining: {len(reviews_clean):,} rows")

# Step 3: Keep only essential columns
print("\n3️⃣ Selecting essential columns...")
reviews_clean = reviews_clean[['author_id', 'rating', 'product_id', 'product_name', 'brand_name', 'price_usd']]
reviews_clean.rename(columns={'author_id': 'user_id'}, inplace=True)

# Step 4: Remove missing product_ids
print("\n4️⃣ Removing missing product IDs...")
initial_count = len(reviews_clean)

reviews_clean = reviews_clean[reviews_clean['product_id'].notna()]

print(f"  Removed {initial_count - len(reviews_clean):,} rows")
print(f"  Remaining: {len(reviews_clean):,} rows")

print(f"\n✅ Cleaned reviews DataFrame:")
print(f"  Shape: {reviews_clean.shape}")
print(f"  Columns: {list(reviews_clean.columns)}")
display(reviews_clean.head())


DATA CLEANING & PREPROCESSING

1️⃣ Cleaning ratings...
  Removed 0 rows with invalid ratings
  Remaining: 1,094,411 rows

2️⃣ Removing duplicate reviews...
  Removed 5,520 duplicate reviews
  Remaining: 1,088,891 rows

3️⃣ Selecting essential columns...

4️⃣ Removing missing product IDs...
  Removed 0 rows
  Remaining: 1,088,891 rows

✅ Cleaned reviews DataFrame:
  Shape: (1088891, 6)
  Columns: ['user_id', 'rating', 'product_id', 'product_name', 'brand_name', 'price_usd']


,user_id,rating,product_id,product_name,brand_name,price_usd
0,1741593524,5,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0
1,31423088263,1,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
2,5061282401,5,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
3,6083038851,5,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0
4,47056667835,5,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0


---

## 3. Filter Sparse Users and Products

In [5]:
print("\n" + "="*60)
print("FILTERING SPARSE DATA")
print("="*60)

# Filter sparse users
print("\n1️⃣ Filtering sparse users...")
MIN_USER_REVIEWS = 3
initial_users = reviews_clean['user_id'].nunique()

user_review_counts = reviews_clean['user_id'].value_counts()
active_users = user_review_counts[user_review_counts >= MIN_USER_REVIEWS].index

reviews_clean = reviews_clean[reviews_clean['user_id'].isin(active_users)].copy()
remaining_users = reviews_clean['user_id'].nunique()

print(f"  Removed users with < {MIN_USER_REVIEWS} reviews")
print(f"  {initial_users:,} → {remaining_users:,} users ({100*(remaining_users/initial_users):.1f}%)")
print(f"  Rows: {len(reviews_clean):,}")

# Filter sparse products
print("\n2️⃣ Filtering sparse products...")
MIN_PRODUCT_REVIEWS = 5
initial_products = reviews_clean['product_id'].nunique()

product_review_counts = reviews_clean['product_id'].value_counts()
popular_products = product_review_counts[product_review_counts >= MIN_PRODUCT_REVIEWS].index

reviews_clean = reviews_clean[reviews_clean['product_id'].isin(popular_products)].copy()
remaining_products = reviews_clean['product_id'].nunique()

print(f"  Removed products with < {MIN_PRODUCT_REVIEWS} reviews")
print(f"  {initial_products:,} → {remaining_products:,} products ({100*(remaining_products/initial_products):.1f}%)")
print(f"  Rows: {len(reviews_clean):,}")

print(f"\n✅ Final filtered dataset: {len(reviews_clean):,} reviews")
print(f"  Active users: {reviews_clean['user_id'].nunique():,}")
print(f"  Popular products: {reviews_clean['product_id'].nunique():,}")


FILTERING SPARSE DATA

1️⃣ Filtering sparse users...
  Removed users with < 3 reviews
  578,653 → 92,138 users (15.9%)
  Rows: 487,840

2️⃣ Filtering sparse products...
  Removed products with < 5 reviews
  2,218 → 1,865 products (84.1%)
  Rows: 487,057

✅ Final filtered dataset: 487,057 reviews
  Active users: 92,132
  Popular products: 1,865


---

## 4. Merge Reviews with Product Metadata

In [6]:
print("\n" + "="*60)
print("MERGING WITH PRODUCT METADATA")
print("="*60)

# Select relevant columns from products (exclude 'rating' to avoid conflict with review ratings)
products_subset = products_df[['product_id', 'product_name', 'brand_name', 'price_usd', 'primary_category']].copy()
products_subset = products_subset.drop_duplicates(subset=['product_id'])

print(f"\nProduct metadata to merge: {len(products_subset):,} unique products")

# Merge reviews with products on product_id
combined_df = pd.merge(reviews_clean, products_subset, on='product_id', how='left', suffixes=('_review', '_product'))

print(f"\n✅ Merged DataFrame shape: {combined_df.shape}")
print(f"Columns: {list(combined_df.columns)}")

# Use product name from product metadata if available
combined_df['product_name'] = combined_df['product_name_product'].fillna(combined_df['product_name_review'])
combined_df = combined_df.drop(columns=['product_name_review', 'product_name_product'])

print(f"\nFinal merged columns: {list(combined_df.columns)}")

# Verify merge
print(f"\n✅ Final Statistics:")
print(f"  Total reviews: {len(combined_df):,}")
print(f"  Unique users: {combined_df['user_id'].nunique():,}")
print(f"  Unique products: {combined_df['product_id'].nunique():,}")
print(f"  Average reviews per user: {len(combined_df) / combined_df['user_id'].nunique():.1f}")
print(f"  Average reviews per product: {len(combined_df) / combined_df['product_id'].nunique():.1f}")

print(f"\nRating distribution:")
rating_dist = combined_df['rating'].value_counts().sort_index()
for rating, count in rating_dist.items():
    pct = 100 * count / len(combined_df)
    print(f"  {int(rating)} stars: {count:,} ({pct:.1f}%)")

display(combined_df.head())


MERGING WITH PRODUCT METADATA

Product metadata to merge: 8,494 unique products

✅ Merged DataFrame shape: (487057, 10)
Columns: ['user_id', 'rating', 'product_id', 'product_name_review', 'brand_name_review', 'price_usd_review', 'product_name_product', 'brand_name_product', 'price_usd_product', 'primary_category']

Final merged columns: ['user_id', 'rating', 'product_id', 'brand_name_review', 'price_usd_review', 'brand_name_product', 'price_usd_product', 'primary_category', 'product_name']

✅ Final Statistics:
  Total reviews: 487,057
  Unique users: 92,132
  Unique products: 1,865
  Average reviews per user: 5.3
  Average reviews per product: 261.2

Rating distribution:
  1 stars: 19,424 (4.0%)
  2 stars: 20,985 (4.3%)
  3 stars: 38,310 (7.9%)
  4 stars: 93,978 (19.3%)
  5 stars: 314,360 (64.5%)


,user_id,rating,product_id,brand_name_review,price_usd_review,brand_name_product,price_usd_product,primary_category,product_name
0,1238130325,4,P420652,LANEIGE,24.0,LANEIGE,24.0,Skincare,Lip Sleeping Mask Intense Hydration with Vitam...
1,2795584162,5,P420652,LANEIGE,24.0,LANEIGE,24.0,Skincare,Lip Sleeping Mask Intense Hydration with Vitam...
2,27991208736,3,P420652,LANEIGE,24.0,LANEIGE,24.0,Skincare,Lip Sleeping Mask Intense Hydration with Vitam...
3,9467587295,5,P420652,LANEIGE,24.0,LANEIGE,24.0,Skincare,Lip Sleeping Mask Intense Hydration with Vitam...
4,12367701277,5,P420652,LANEIGE,24.0,LANEIGE,24.0,Skincare,Lip Sleeping Mask Intense Hydration with Vitam...


---

## 5. Export Processed Data

In [7]:
print("\n" + "="*60)
print("EXPORTING PROCESSED DATA")
print("="*60)

# Export merged data
output_file = 'output/merged_data.csv'
combined_df.to_csv(output_file, index=False, quoting=csv.QUOTE_MINIMAL)
print(f"\n✅ Exported merged data: {output_file}")
print(f"  File size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")
print(f"  Rows: {len(combined_df):,}")
print(f"  Columns: {len(combined_df.columns)}")

# Export data summary as JSON
data_summary = {
    "total_reviews": int(len(combined_df)),
    "unique_users": int(combined_df['user_id'].nunique()),
    "unique_products": int(combined_df['product_id'].nunique()),
    "avg_reviews_per_user": float(len(combined_df) / combined_df['user_id'].nunique()),
    "avg_reviews_per_product": float(len(combined_df) / combined_df['product_id'].nunique()),
    "global_mean_rating": float(combined_df['rating'].mean()),
    "rating_distribution": {
        int(rating): int(count) 
        for rating, count in combined_df['rating'].value_counts().sort_index().items()
    },
    "min_user_reviews_threshold": int(MIN_USER_REVIEWS),
    "min_product_reviews_threshold": int(MIN_PRODUCT_REVIEWS),
    "columns": list(combined_df.columns)
}

summary_file = 'output/data_summary.json'
with open(summary_file, 'w') as f:
    json.dump(data_summary, f, indent=2)

print(f"\n✅ Exported data summary: {summary_file}")

# Export product mapping for recommendations notebook
product_mapping = {
    'product_to_id': {pid: i for i, pid in enumerate(combined_df['product_id'].unique())},
    'id_to_product': {i: pid for i, pid in enumerate(combined_df['product_id'].unique())},
    'total_products': int(combined_df['product_id'].nunique())
}

mapping_file = 'output/product_mapping.json'
with open(mapping_file, 'w') as f:
    json.dump(product_mapping, f, indent=2)

print(f"✅ Exported product mapping: {mapping_file}")

print("\n" + "="*60)
print("✅ DATA PREPROCESSING COMPLETE!")
print("="*60)
print("\nNext step: Run '2-Recommender-System.ipynb'")
print("="*60)


EXPORTING PROCESSED DATA

✅ Exported merged data: output/merged_data.csv
  File size: 49.35 MB
  Rows: 487,057
  Columns: 9

✅ Exported data summary: output/data_summary.json
✅ Exported product mapping: output/product_mapping.json

✅ DATA PREPROCESSING COMPLETE!

Next step: Run '2-Recommender-System.ipynb'
